# My Workout Analytics (FitNotes Dataset)

1. Import libraries
2. Data overview
3. Data cleanup ?
4. Feature engineering
5. Statistics & Analytics

In [27]:
import pandas as pd
import numpy as np

This notebook begins by loading the latest FitNotes export and preparing it for analysis.

In [28]:
# Load CSV
def load_fitnotes_csv(filename=None):
    from glob import glob
    from os import path
    # Get latest csv
    dataset_filename = max(glob("FitNotes_Export_*.csv"), key=path.getmtime) if filename is None else filename
    # Return as pandas dataframe
    return pd.read_csv(dataset_filename)

df = load_fitnotes_csv()

# Overview
df.tail()

,Date,Exercise,Category,Weight,Weight Unit,Reps,Distance,Distance Unit,Time,Comment
4061,2026-06-05,Decline Sit Up,Abs,1.0,kgs,9.0,NaN,NaN,NaN,NaN
4062,2026-06-05,Barbell Curl,Biceps,17.5,kgs,12.0,NaN,NaN,NaN,NaN
4063,2026-06-05,Barbell Curl,Biceps,17.5,kgs,8.0,NaN,NaN,NaN,NaN
4064,2026-06-05,Rope Push Down,Triceps,12.5,kgs,15.0,NaN,NaN,NaN,NaN
4065,2026-06-05,Rope Push Down,Triceps,12.5,kgs,12.0,NaN,NaN,NaN,NaN


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4066 entries, 0 to 4065
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           4066 non-null   object 
 1   Exercise       4066 non-null   object 
 2   Category       4066 non-null   object 
 3   Weight         3899 non-null   float64
 4   Weight Unit    3899 non-null   object 
 5   Reps           3894 non-null   float64
 6   Distance       167 non-null    float64
 7   Distance Unit  167 non-null    object 
 8   Time           172 non-null    object 
 9   Comment        176 non-null    object 
dtypes: float64(3), object(7)
memory usage: 317.8+ KB


## Feature Engineering

While FitNotes provides only basic exercise categorization, we can enrich the dataset by adding our own classifications, such as primary and secondary muscle groups, movement patterns (push, pull, hinge, squat), and whether an exercise uses external load or bodyweight.

### Add Exercise metadata

First, create an Exercise catalog:

In [30]:
exercise_catalog = sorted(df["Exercise"].unique())

print(len(exercise_catalog), "total exercises:")

for exercise in exercise_catalog:
   print(exercise)

197 total exercises:
1.5 Back Squat
90° Nordic Curl
Adv. Tuck/L Front Lever Row
Alisa Squat
Alternating Bulgarian Split Squat
Alternating Single Leg Curl
Alternating Single Leg Extension
Archer Pull Up Negative
Archer Push Up
Archer Ring Push Up
Archer Ring Row
Archer Squat
BW Bar Overhead Tricep Extension
BW Bicep Curl
BW Face Pull
Back Lever Prog.
Band Assisted Muscle Up
Barbell 21s Curl
Barbell Calf Raise
Barbell Curl
Barbell Front Squat
Barbell Hack Squat
Barbell Hip Thrust
Barbell Row
Barbell Squat
Barbell Triceps Overhead Extension
Biking
Bottom Squat Hold
Bulgarian Split Squat
Cable Crunch
Cable Face Pull
Cable Hammer Curl
Cable Lateral Raise
Cable Overhead Triceps Extension
Calf Raise
Chest Press Machine
Chin Up
Clamshell
Climbing
Copenhagen Hip Dip
Copenhagen Plank
Crab Walks
Crunch
Curtsy Squat
DB Romanian Deadlift
DB Single-leg RDL
Deadlift
Decline Pike Push Up
Decline Push Up
Decline Ring Push Up
Decline Sit Up
Deficit Bulgarian Split Squat
Deficit Decline Push Up
Deficit P

In [31]:
# Now add empty metadata columns
catalog_df = pd.DataFrame({"Exercise": exercise_catalog})

catalog_df['muscles_pimary'] = "" # primary muscle group exercised
catalog_df['muscles_secondary'] = "" # secondary muscle group worked
catalog_df['exercise_complexity'] = "" # compound or isolation 
catalog_df['exercise_level'] = "" # beginner, intermediate, advanced
catalog_df['movement_pattern'] = "" # push pull squat hinge, lunge; see table
catalog_df['body_region'] = "" # upper, lower, core, fullbody
catalog_df['skill_category'] = "" # strength foundation, handstand, muscle up, planche, font lever (group progressions)
catalog_df['laterallity'] = "" # uni, bi
catalog_df['resistance_type'] = "" # bodyweight, free weights, machine, band
catalog_df['equipment'] = "" # bar, rings, parallel bars, floor, db, barbell, machine, cable, band

# Export
catalog_df.to_csv("exercise_metadata_empty.csv", index=False)


Open that CSV and edit manually using a spreadsheet software.

Fill it like:
| Exercise      | muscle_primary | movement_pattern | resistance_type |
| ------------- | -------------- | ---------------- | --------------- |
| Pull Up       | Lats           | Vertical Pull    | Bodyweight      |
| Dip           | Chest          | Vertical Push    | Bodyweight      |
| Barbell Squat | Quads          | Squat            | Free Weight     |

Delete any exercises you don't care about.

Save as `exercise_metadata.csv`

In [32]:
# Load it back
metadata = pd.read_csv("exercise_metadata.csv")

# Compare / diff
df_exercises = set(df["Exercise"].unique())
metadata_exercises = set(metadata["Exercise"].unique())

# In df but not in metadata
missing_in_metadata = df_exercises - metadata_exercises

# In metadata but not in df
missing_in_df = metadata_exercises - df_exercises

print("Exercises in tracker but NOT in metadata (add them or they will be dropped):")
print(sorted(missing_in_metadata))

print("\nExercises in metadata but NOT in tracker (ignore):")
print(sorted(missing_in_df))

Exercises in tracker but NOT in metadata (add them or they will be dropped):
['Biking', 'Climbing', 'Running (Outdoor)', 'Running (Treadmill)']

Exercises in metadata but NOT in tracker (ignore):
[]


If everything is OK, proceed to merge into the main dataset.

In [33]:
# Clean up metadata table
metadata = metadata[metadata["Exercise"].isin(df_exercises)]

# Merge into tracker df
df = df.merge(metadata, on="Exercise", how="right")

# Drop exercises without metadata
df = df[df["Exercise"].isin(metadata_exercises)]

# New overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4060 entries, 0 to 4059
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 4060 non-null   object 
 1   Exercise             4060 non-null   object 
 2   Category             4060 non-null   object 
 3   Weight               3899 non-null   float64
 4   Weight Unit          3899 non-null   object 
 5   Reps                 3894 non-null   float64
 6   Distance             161 non-null    float64
 7   Distance Unit        161 non-null    object 
 8   Time                 166 non-null    object 
 9   Comment              176 non-null    object 
 10  muscles_primary      4060 non-null   object 
 11  muscles_secondary    3018 non-null   object 
 12  exercise_complexity  4060 non-null   object 
 13  exercise_level       4060 non-null   object 
 14  movement_pattern     3044 non-null   object 
 15  body_region          4060 non-null   o

### Derived features

Now we can add new features based on the existing data.

In [34]:
# First get the real weight of bodyweight exercises:
BODYWEIGHT = 53

df["real_weight"] = np.where(
    (df["resistance_type"] == "bodyweight") & (df['Weight'].notna()),
    np.where(
        # if Weight > 1.0:
        df["Weight"] > 1.0,
        # Use BW + added weight,
        BODYWEIGHT + df["Weight"],
        # Else: use BW only.
        BODYWEIGHT
    ),
    df["Weight"]
)

df[df['resistance_type'] == "bodyweight"]

,Date,Exercise,Category,Weight,Weight Unit,Reps,Distance,Distance Unit,Time,Comment,...,muscles_secondary,exercise_complexity,exercise_level,movement_pattern,body_region,skill_category,laterallity,resistance_type,equipment,real_weight
0,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
1,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
2,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,11.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
3,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
4,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4051,2026-05-14,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,0:00:10,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4052,2026-05-14,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,0:00:10,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4053,2026-06-01,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,0:00:10,3m rest,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4054,2026-06-01,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,0:00:12,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN


In [35]:

#df['intensity_zone'] = "" # 1-5 max str, 6-12 str/hypr, 13-20 hypr/endur, 20+ endur.

With this added structure, we can analyze training volume over time, track muscle-group balance, measure progression in load or reps, examine consistency and frequency, and explore trends such as intensity cycles or recovery gaps. The goal is to transform simple workout logs into a structured dataset that reveals meaningful patterns in training behavior and performance.